# Capítulo 3 - Exercício 2: Aumento de Dados / Expansão do Conjunto de Treinamento

O objetivo deste notebook é melhorar a acurácia do classificador MNIST usando aumento de dados. Para cada imagem de treino, criamos quatro cópias deslocadas em 1 pixel: para baixo, para cima, para a direita e para a esquerda. Depois treinamos o melhor KNN do exercício anterior no conjunto aumentado e avaliamos no conjunto de teste original.

## Plano do exercício

1. Carregar os dados do MNIST.
2. Separar treino e teste usando a divisão padrão do dataset.
3. Criar uma função genérica para deslocar uma imagem MNIST.
4. Expandir o conjunto de treino com quatro cópias deslocadas por imagem.
5. Embaralhar o treino aumentado.
6. Treinar o KNN com os melhores hiperparâmetros encontrados no exercício anterior.
7. Avaliar a acurácia final no conjunto de teste original.

## Configuração

Importamos as bibliotecas usadas no exercício e verificamos versões mínimas. Mantemos uma semente fixa para tornar os resultados mais reprodutíveis.

In [1]:
%%time
import sys
import numpy as np

from packaging import version
import sklearn

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

np.random.seed(42)

CPU times: user 3.37 s, sys: 137 ms, total: 3.51 s
Wall time: 1.04 s


### Medição de tempo

As células de código usam `%%time` para mostrar quanto tempo cada etapa levou. O valor mais importante é `Wall time`, que representa o tempo real percebido até a célula terminar.

# MNIST

In [2]:
%%time
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', as_frame=False)

CPU times: user 2.06 s, sys: 359 ms, total: 2.42 s
Wall time: 2.46 s


### Conferindo os dados

In [3]:
%%time
X, y = mnist.data, mnist.target

X.shape, y.shape, np.unique(y)

CPU times: user 16.3 ms, sys: 0 ns, total: 16.3 ms
Wall time: 16.4 ms


((70000, 784),
 (70000,),
 array(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], dtype=object))

### Dividindo entre dados de treino e teste

In [4]:
%%time
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

CPU times: user 10 μs, sys: 1 μs, total: 11 μs
Wall time: 13.1 μs


### Aumentando o conjunto de treino

O exercício pede deslocamentos de 1 pixel. Também reconstruímos o treino a partir de `X[:60000]` e `y[:60000]` para evitar multiplicar o dataset novamente caso esta célula seja executada mais de uma vez.

In [5]:
%%time
from scipy.ndimage import shift

SHIFT_PIXELS = 1

def shift_image(img_data, row_shift, col_shift):
    img = img_data.reshape(28, 28)
    shifted_img = shift(
        img,
        [row_shift, col_shift],
        cval=0,
        mode="constant",
        order=0,
    )
    return shifted_img.reshape(-1)

X_train_original = X[:60000].copy()
y_train_original = y[:60000].copy()

X_train_augmented = [image for image in X_train_original]
y_train_augmented = [label for label in y_train_original]

for image, label in zip(X_train_original, y_train_original):
    X_train_augmented.extend([
        shift_image(image, SHIFT_PIXELS, 0),
        shift_image(image, -SHIFT_PIXELS, 0),
        shift_image(image, 0, SHIFT_PIXELS),
        shift_image(image, 0, -SHIFT_PIXELS),
    ])
    y_train_augmented.extend([label] * 4)

X_train = np.array(X_train_augmented)
y_train = np.array(y_train_augmented)

shuffle_idx = np.random.permutation(len(X_train))
X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]

CPU times: user 2.73 s, sys: 624 ms, total: 3.35 s
Wall time: 3.38 s


In [6]:
%%time
X_train.shape

CPU times: user 3 μs, sys: 1 μs, total: 4 μs
Wall time: 3.81 μs


(300000, 784)

# Classificador multiclasse: dígitos de 0 a 9

In [7]:
%%time
from sklearn.neighbors import KNeighborsClassifier

knn_clf_multiclasse = KNeighborsClassifier(n_neighbors=3, weights="distance")
knn_clf_multiclasse.fit(X_train, y_train)

CPU times: user 86.6 ms, sys: 17.5 ms, total: 104 ms
Wall time: 138 ms


KNeighborsClassifier(n_neighbors=3, weights='distance')

## Validação cruzada

A validação cruzada no treino aumentado inteiro fica muito pesada para KNN, porque o conjunto passa de 60.000 para 300.000 imagens. Além disso, validar dados aumentados exige cuidado para não misturar cópias da mesma imagem entre treino e validação. Neste exercício, a comparação mais importante é a avaliação final no conjunto de teste original.

In [8]:
%%time
# Checagem opcional e reduzida, apenas para ter uma noção rápida.
# A métrica decisiva do exercício continua sendo a acurácia no conjunto de teste.
from sklearn.model_selection import StratifiedKFold, cross_val_score

sample_size = 15000
sample_idx = np.random.choice(len(X_train), sample_size, replace=False)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

cross_val_score(
    KNeighborsClassifier(n_neighbors=3, weights="distance"),
    X_train[sample_idx],
    y_train[sample_idx],
    cv=cv,
    scoring="accuracy",
    n_jobs=2,
)

CPU times: user 53.4 ms, sys: 42.7 ms, total: 96.1 ms
Wall time: 2.51 s


array([0.9266, 0.9284, 0.9338])

# Avaliação final no conjunto de teste

O teste não é aumentado. Ele deve representar imagens novas, no formato original do MNIST.

In [9]:
%%time
from sklearn.metrics import accuracy_score

y_test_pred_multiclasse = knn_clf_multiclasse.predict(X_test)
accuracy_score(y_test, y_test_pred_multiclasse)

CPU times: user 6min, sys: 1min, total: 7min
Wall time: 52.5 s


0.9763

## Observação sobre tempo e Grid Search

Não é necessário repetir `GridSearchCV` no treino aumentado completo para concluir este exercício. O conjunto aumentado tem 300.000 imagens, e o KNN fica caro justamente na etapa de previsão. Por isso, reotimizar hiperparâmetros nesse dataset pode custar muito tempo para um ganho pequeno. Aqui usamos os melhores parâmetros já encontrados no exercício anterior: `n_neighbors=3` e `weights="distance"`.